In [1]:
import ast
import pandas as pd

pd.set_option("display.max_columns", 100)

hltv = pd.read_csv("C:/Users/Nico/Documents/The Power MBA/Proyecto Final/cs2-pro-scene-analysis/data/raw/hltv_playerStats-complete.csv")
df = hltv.copy()
print(df.shape)

(803, 20)


In [2]:
print("nulos totales:", df.isna().sum().sum())
print("filas duplicadas:", df.duplicated().sum())
print("nicks repetidos ignorando mayúsculas:", df["nick"].str.lower().duplicated().sum())

nulos totales: 0
filas duplicadas: 0
nicks repetidos ignorando mayúsculas: 5


In [3]:
df["nick"] = df["nick"].str.strip()
df["nick_key"] = df["nick"].str.lower()

df["hltv_player_id"] = df["stats_link"].str.extract(r"/players/(\d+)/")[0].astype("int64")
print("ids repetidos:", df["hltv_player_id"].duplicated().sum())

df = df.drop(columns="stats_link")

ids repetidos: 0


In [4]:
iso = {
    "Argentina": "AR", "Australia": "AU", "Belarus": "BY", "Belgium": "BE",
    "Bosnia and Herzegovina": "BA", "Brazil": "BR", "Bulgaria": "BG", "Canada": "CA",
    "China": "CN", "Czech Republic": "CZ", "Denmark": "DK", "Estonia": "EE",
    "Finland": "FI", "France": "FR", "Germany": "DE", "Guatemala": "GT",
    "Hong Kong": "HK", "Hungary": "HU", "Indonesia": "ID", "Israel": "IL",
    "Jordan": "JO", "Kazakhstan": "KZ", "Korea": "KR", "Kosovo": "XK",
    "Latvia": "LV", "Lithuania": "LT", "Malaysia": "MY", "Mexico": "MX",
    "Mongolia": "MN", "Montenegro": "ME", "Netherlands": "NL", "New Zealand": "NZ",
    "North Macedonia": "MK", "Norway": "NO", "Poland": "PL", "Portugal": "PT",
    "Romania": "RO", "Russia": "RU", "Serbia": "RS", "Singapore": "SG",
    "Slovakia": "SK", "South Africa": "ZA", "Spain": "ES", "Sweden": "SE",
    "Switzerland": "CH", "Taiwan": "TW", "Thailand": "TH", "Turkey": "TR",
    "Ukraine": "UA", "United Kingdom": "GB", "United States": "US",
    "Uruguay": "UY", "Uzbekistan": "UZ",
}

df = df.rename(columns={"country": "country_name"})
df["country_code"] = df["country_name"].map(iso)
print("países sin código:", df["country_code"].isna().sum())

países sin código: 0


In [5]:
df["num_teams"] = df["teams"].apply(lambda s: len(ast.literal_eval(s)))
print(df["num_teams"].value_counts().sort_index().to_dict())

{1: 112, 2: 267, 3: 285, 4: 119, 5: 17, 6: 3}


In [6]:
calculado = df["total_kills"] - df["total_deaths"]
print("filas que no cuadran:", (df["kd_difference"] != calculado).sum())

df["kd_difference"] = calculado
print("tras corregir:", (df["kd_difference"] != calculado).sum())

filas que no cuadran: 28
tras corregir: 0


In [7]:
df = df.rename(columns={
    "maps_played": "career_maps", "rounds_played": "career_rounds",
    "kd_difference": "career_kd_diff", "kd_ratio": "career_kd_ratio",
    "rating": "career_rating", "total_kills": "career_kills",
    "headshot_percentage": "career_hs_pct", "total_deaths": "career_deaths",
    "grenade_damage_per_round": "career_grenade_dpr",
    "kills_per_round": "career_kpr", "assists_per_round": "career_apr",
    "deaths_per_round": "career_dpr",
    "teammate_saved_per_round": "career_saves_pr",
    "saved_by_teammate_per_round": "career_saved_by_pr",
    "kast": "career_kast", "impact": "career_impact",
})

In [8]:
print("repetidos en nick_key + country_code:", df.duplicated(["nick_key", "country_code"]).sum())
print(df.shape)
print("mínimo de mapas por jugador:", df["career_maps"].min())

df.to_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\raw\\hltv_players_clean.csv", index=False)

repetidos en nick_key + country_code: 0
(803, 23)
mínimo de mapas por jugador: 374
